**Statsmodels** is a Python library for statistical modeling and analysis.  
It provides tools for time series analysis, regression, and hypothesis testing.  
In this notebook, it's used to test stationarity (ADF test) and plot autocorrelation (ACF/PACF) for time series forecasting.

In [0]:
%pip install statsmodels

In [0]:
# Cell 1: Load silver and visualize
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

pdf = spark.table("weather_silver").toPandas()
pdf = pdf.sort_values("date").set_index("date")
ts = pdf["avg_temp"]



In [0]:
display(ts)

In [0]:
# Cell 2: Time series plot (display in notebook)
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(ts, linewidth=0.8, color="#1D9E75")
ax.set_title("Daily Average Temperature — Chennai 2022–2023")
ax.set_ylabel("Temperature (°C)")
display(fig)  # Databricks renders matplotlib via display()



In [0]:
# Cell 3: ADF Stationarity Test
result = adfuller(ts.dropna())

print(f"ADF Statistic : {result[0]:.4f}")
print(f"p-value       : {result[1]:.4f}")
print("Stationary" if result[1] < 0.05 else "Non-stationary → needs differencing")

display(result)

In [0]:
# Cell 4: ACF and PACF plots (crucial for choosing p, q)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_acf(ts.dropna(), lags=40, ax=axes[0])
plot_pacf(ts.dropna(), lags=40, ax=axes[1])
display(fig)


The ACF and PACF plots above reveal the distribution and structure of the daily average temperature time series:

- The **ACF plot** shows a slow decay, indicating strong autocorrelation and suggesting the data is not purely random. This pattern is typical of time series with trend or seasonality.
- The **PACF plot** displays significant spikes at the first few lags, implying that recent past values have a strong influence on the current value.

Overall, the data distribution is not stationary and exhibits temporal dependence, meaning values are correlated over time. This is common in weather data, where temperatures follow seasonal patterns and trends.

The above plot shows the Autocorrelation Function (ACF) and Partial Autocorrelation Function (PACF) for the daily average temperature time series.  
- **ACF (left):** Measures how each value in the series is correlated with its previous values (lags). Peaks indicate significant autocorrelation at those lags.
- **PACF (right):** Shows the correlation between the series and its lags, after removing the influence of intermediate lags. Peaks help identify the order of autoregressive (AR) terms for time series modeling.

These plots are used to determine the appropriate parameters for ARIMA models (p and q).